# Source-Aware OSINT Agent with Pydantic AI

The presentation shows a more production-like system: case folders, `AGENTS.md`, an investigation playbook, separate skills, logs, and reports. This notebook keeps the same idea, but compresses it into one Colab-friendly flow:

**case input → source-aware agent → registry/database tools + Pydantic provider-native web search → structured report**


## How this maps to the presentation


| In the presentation | In this notebook |
|---|---|
| `AGENTS.md` | The agent instructions cell |
| `INVESTIGATION_PLAYBOOK.md` | Tool docstrings + workflow rules |
| `skills/*/SKILL.md` | Custom Python tools such as YC World and Aleph lookup |
| Orchestrator skill | The Pydantic AI `Agent` deciding what to call |
| Case folder / `CASE.md` | The company + question prompt |
| Final investigation report | The `ResearchReport` Pydantic output model |

The biggest simplification: we do **not** write a folder-based state machine here. We let Pydantic AI handle provider-native web search and tool calling, then we force the answer into a structured report.


In [ ]:
# Prompt: Create a setup code cell that installs the packages needed to run Pydantic AI with OpenAI and HTTP-based source tools in Colab.
# Setup cell
%pip install -q requests==2.32.5 "pydantic-ai-slim[openai]==0.8.1" --quiet


In [ ]:
# Prompt: Create a configuration code cell that imports libraries, loads API keys from .env or secure prompts, defines source URLs, and prints enabled services.
# Paste keys into the empty strings 

import json
import os
import re
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from typing import Literal

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.native_tools import WebSearchTool
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.settings import ModelSettings

OPENAI_API_KEY = ""
YC_WORLD_API_KEY = ""
ALEPH_API_KEY = ""
OPENAI_MODEL = "gpt-4.1-mini"
YC_WORLD_BASE_URL = "https://api.youcontrol.world"
ALEPH_BASE_URL = "https://aleph.occrp.org"
OUTPUT_DIR = Path("outputs")

load_dotenv(Path(".env"))

for key, value in [
    ("OPENAI_API_KEY", OPENAI_API_KEY),
    ("YC_WORLD_API_KEY", YC_WORLD_API_KEY),
    ("ALEPH_API_KEY", ALEPH_API_KEY),
]:
    if value:
        os.environ[key] = value

for key, label in [
    ("OPENAI_API_KEY", "OpenAI API key"),
    ("YC_WORLD_API_KEY", "YC World API key (Enter to skip)"),
    ("ALEPH_API_KEY", "Aleph API key (Enter to skip; public search may still work)"),
]:
    if not os.getenv(key):
        value = getpass(f"{label}: ")
        if value:
            os.environ[key] = value

OPENAI_MODEL = os.getenv("OPENAI_MODEL", OPENAI_MODEL)
YC_WORLD_API_KEY = os.getenv("YC_WORLD_API_KEY", YC_WORLD_API_KEY)
ALEPH_API_KEY = os.getenv("ALEPH_API_KEY", ALEPH_API_KEY) or os.getenv("ALEPHCLIENT_API_KEY", "")
YC_WORLD_BASE_URL = os.getenv("YC_WORLD_BASE_URL", YC_WORLD_BASE_URL).rstrip("/")
ALEPH_BASE_URL = os.getenv("ALEPH_BASE_URL", ALEPH_BASE_URL).rstrip("/")

print(f"Model: OpenAI Responses API / {OPENAI_MODEL}")
print(f"YC World: {'enabled' if YC_WORLD_API_KEY else 'disabled'}")
print(f"Aleph: {'key provided' if ALEPH_API_KEY else 'public/no-key mode'}")


## Report Shape

This cell is like the **report template**.

Pydantic forces the model to separate facts, claims, leads, hypotheses, and discourages one big blurry answer.


In [ ]:
# Prompt: Create a code cell defining a Pydantic ResearchReport model with registry_records, public_findings, and source_ledger, including what belongs in each field.

class ResearchReport(BaseModel):
    registry_records: list[str] = Field(default_factory=list, description="Concrete registry rows with company IDs, dates, roles, source IDs, and registry_evidence.json path.")
    public_findings: list[str] = Field(default_factory=list, description="Findings from public sources such as journalistic articles, investigations, media reports, NGO reports, public records, and company websites.")
    source_ledger: list[str] = Field(default_factory=list, description="URLs, source names, record IDs, database names, and registry_evidence.json path used.")


## Custom Tools

This cell is like the **skills folder**.

The notebook now has only special-source tools here. General web search is exposed through Pydantic AI's provider-native `WebSearchTool`.


In [ ]:
# Prompt: Create a code cell that manages case output folders, extracts list-like source fields, and writes one registry_evidence.json file.

CASE_OUTPUT_DIR = None
EVIDENCE_ENTRIES = []


def slug(text: str) -> str:
    return re.sub(r"[^0-9a-zA-Z]+", "_", text.lower()).strip("_") or "company"


def case_folder(company: str) -> Path:
    return OUTPUT_DIR / slug(company)


def registry_evidence_path() -> Path:
    folder = CASE_OUTPUT_DIR or OUTPUT_DIR / "_uncategorized"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / "registry_evidence.json"


def write_registry_evidence() -> Path:
    path = registry_evidence_path()
    payload = {
        "updated_at": datetime.now(timezone.utc).isoformat(),
        "entries": EVIDENCE_ENTRIES,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def set_case_evidence_file(company: str) -> Path:
    global CASE_OUTPUT_DIR, EVIDENCE_ENTRIES
    CASE_OUTPUT_DIR = case_folder(company)
    CASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    EVIDENCE_ENTRIES = []
    return write_registry_evidence()


def add_registry_evidence(source: str, kind: str, request: dict, data: dict) -> Path:
    EVIDENCE_ENTRIES.append({"source": source, "kind": kind, "request": request, "data": data})
    return write_registry_evidence()


def prop(props: dict, key: str, limit: int = 3) -> list[str]:
    value = props.get(key)
    if isinstance(value, list):
        return [str(item) for item in value[:limit] if item not in (None, "")]
    return [str(value)] if value not in (None, "") else []


def text(props: dict, key: str, limit: int = 3) -> str:
    return ", ".join(prop(props, key, limit))


In [ ]:
# Prompt: Create a code cell with direct YC World tools that search entities, fetch detailed relations, pull key fields, and write one evidence JSON file. follow the documentation https://api.youcontrol.world/swagger/index.html

ID_FIELDS = {
    "registrationNumber": "registration number",
    "taxNumber": "tax number",
    "vatCode": "VAT",
    "innCode": "INN",
    "ogrnCode": "OGRN",
    "leiCode": "LEI",
    "kppCode": "KPP",
}


def yc_entity_name(entity: dict) -> str:
    item = (entity.get("items") or [{}])[0]
    props = item.get("properties", {}) or {}
    return text(props, "name", 4) or item.get("caption") or entity.get("caption") or "unknown"


def yc_entity_line(label: str, entity: dict, external_id: str = "") -> str:
    item = (entity.get("items") or [{}])[0]
    props = item.get("properties", {}) or {}
    schema = item.get("schema") or entity.get("schema") or "Entity"
    bits = [f"{label}: {yc_entity_name(entity)}", f"schema: {schema}"]

    if schema.lower() not in {"person", "human"}:
        for key, name in ID_FIELDS.items():
            value = text(props, key)
            if value:
                bits.append(f"{name}: {value}")

    for key, name in [
        ("country", "country"),
        ("jurisdiction", "jurisdiction"),
        ("legalForm", "legal form"),
        ("incorporationDate", "incorporated"),
        ("status", "status"),
        ("address", "address"),
        ("sourceUrl", "source URL"),
        ("publisher", "publisher"),
    ]:
        value = text(props, key, 2 if key == "address" else 3)
        if value:
            bits.append(f"{name}: {value}")

    if external_id:
        bits.append(f"YC World ID: {external_id}")
    return "; ".join(bits)


def yc_entity_record(entity: dict, external_id: str = "") -> dict:
    item = (entity.get("items") or [{}])[0]
    props = item.get("properties", {}) or {}
    schema = item.get("schema") or entity.get("schema") or "Entity"
    return {
        "name": yc_entity_name(entity),
        "schema": schema,
        "yc_world_id": external_id,
        "ids": {label: prop(props, key) for key, label in ID_FIELDS.items() if prop(props, key)},
        "country": prop(props, "country"),
        "jurisdiction": prop(props, "jurisdiction"),
        "legal_form": prop(props, "legalForm"),
        "incorporated": text(props, "incorporationDate", 1),
        "status": text(props, "status", 1),
        "address": prop(props, "address", 2),
        "source_url": text(props, "sourceUrl", 1),
        "publisher": text(props, "publisher", 1),
    }


def yc_world_entity_detail(external_id: str) -> str:
    """
    Fetch one YC World entity with /Entity/{externalId}/get-entity.

    Use after yc_world_lookup when you need directors, founders, owners,
    authorized signatories, related companies, relation roles, or relation dates.
    """
    request = {"url": f"{YC_WORLD_BASE_URL}/Entity/{external_id}/get-entity"}
    response = requests.get(
        request["url"],
        headers={"Accept": "application/json", "x-api-key": YC_WORLD_API_KEY},
        timeout=45,
    )
    response.raise_for_status()
    raw = response.json()
    data = raw.get("result", raw)

    main_entity = {"items": data.get("items", [])}
    evidence = {
        "main_entity": yc_entity_record(main_entity, external_id),
        "relation_counts": data.get("relationsCount") or [],
        "relations": [],
    }

    lines = [yc_entity_line("Main entity", main_entity, external_id)]
    counts = data.get("relationsCount") or []
    if counts:
        lines.append("Relation counts: " + ", ".join(f"{row.get('schema', 'unknown')}={row.get('count', '?')}" for row in counts))

    for related in (data.get("relationsData") or [])[:12]:
        relation_data = related.get("relationData", {}) or {}
        related_record = yc_entity_record(related, related.get("externalId", ""))
        lines.append(yc_entity_line("Related entity", related, related.get("externalId", "")))

        for relation_item in (relation_data.get("items") or [])[:4]:
            props = relation_item.get("properties", {}) or {}
            relation_record = {
                "related_entity": related_record,
                "relation_schema": relation_data.get("schema") or relation_item.get("schema", "related"),
                "role": prop(props, "role", 4),
                "ownership_percent": prop(props, "percentage", 2),
                "shares_value": prop(props, "sharesValue", 4),
                "start_date": text(props, "startDate", 1),
                "end_date": text(props, "endDate", 1),
                "source_url": text(props, "sourceUrl", 1),
                "publisher": text(props, "publisher", 1),
                "description": prop(props, "description", 4),
                "summary": prop(props, "summary", 4),
            }
            evidence["relations"].append(relation_record)

            relation_bits = [f"relation schema: {relation_record['relation_schema']}"]
            for key, label in [
                ("role", "role"),
                ("ownership_percent", "ownership"),
                ("shares_value", "shares value"),
                ("start_date", "start date"),
                ("end_date", "end date"),
                ("source_url", "source URL"),
                ("publisher", "publisher"),
            ]:
                value = relation_record[key]
                value = ", ".join(value) if isinstance(value, list) else value
                if value:
                    relation_bits.append(f"{label}: {value}")
            lines.append("  Relation detail: " + "; ".join(relation_bits))

    evidence_path = add_registry_evidence("YC World", "entity_detail", request, evidence)
    return f"Evidence JSON: {evidence_path}\n" + "\n".join(lines)


def yc_world_lookup(query: str, schema: Literal["Company", "Person"] = "Company", country: str = "") -> str:
    """
    Search YC World registry records and fetch detail for the strongest matches.

    Use for registry facts, company IDs, incorporation dates, directors, owners,
    authorized signatories, related companies, relation roles, and relation dates.
    """
    country_code = (country or "").strip().lower()
    params = {"SearchString": query, "SchemaName": schema, "PageSize": 3, "Offset": 0}
    if country_code:
        params["Countries"] = country_code

    request = {"url": f"{YC_WORLD_BASE_URL}/GetEntities", "params": params}
    response = requests.get(
        request["url"],
        params=params,
        headers={"Accept": "application/json", "x-api-key": YC_WORLD_API_KEY},
        timeout=45,
    )
    response.raise_for_status()
    raw = response.json()
    payload = raw.get("result", raw)
    entities = payload.get("entities", [])[:3]

    evidence_path = add_registry_evidence(
        "YC World",
        "entity_search",
        request,
        {
            "query": query,
            "schema": schema,
            "country": country_code or "any",
            "total_results": payload.get("total", 0),
            "entities": [yc_entity_record(entity, entity.get("externalId", "")) for entity in entities],
        },
    )

    lines = [
        f"Evidence JSON: {evidence_path}",
        f"YC World search: {query}; schema: {schema}; country: {country_code or 'any'}; total results: {payload.get('total', 0)}",
    ]

    for position, entity in enumerate(entities):
        external_id = entity.get("externalId", "")
        lines.append(yc_entity_line("Search result", entity, external_id))
        if external_id and position < 2:
            lines.append(yc_world_entity_detail(external_id))
        lines.append("")

    if not entities:
        lines.append("No YC World results found.")
    return "\n".join(lines)


In [ ]:
# Prompt: Create a code cell with an Aleph search tool that returns readable entity results and appends normalized results to the evidence JSON file.

# Aleph = another special-source skill.
def aleph_lookup(query: str, schema: Literal["Company", "Person"] = "Company") -> str:
    """
    Search Aleph. Use schema='Company' or schema='Person'.

    Use for investigative datasets, sanctions-style records, leaks,
    PEP-style links, cross-border leads, and related entities.

    Include Aleph URLs, entity IDs, dataset names, and registry_evidence.json in the report ledger.
    """
    headers = {"Accept": "application/json"}
    if ALEPH_API_KEY:
        headers["Authorization"] = f"ApiKey {ALEPH_API_KEY}"

    params = {"q": query, "filter:schema": schema, "limit": 5}
    request_metadata = {"url": f"{ALEPH_BASE_URL}/api/2/entities", "params": params}
    response = requests.get(
        request_metadata["url"],
        params=params,
        headers=headers,
        timeout=45,
    )
    response.raise_for_status()
    raw_response = response.json()
    rows = raw_response.get("results", [])[:5]
    evidence_path = add_registry_evidence(
        "Aleph",
        "entity_search",
        request_metadata,
        {
            "query": query,
            "schema": schema,
            "total_results": raw_response.get("total", 0),
            "entities": [
                {
                    "name": text(item.get("properties") or {}, "name") or item.get("caption") or item.get("id"),
                    "schema": item.get("schema", ""),
                    "collection": (item.get("collection") or {}).get("label", ""),
                    "countries": item.get("countries"),
                    "properties": item.get("properties", {}),
                    "aleph_id": item.get("id", ""),
                    "url": (item.get("links") or {}).get("ui", ""),
                }
                for item in rows
            ],
        },
    )

    lines = [f"Evidence JSON: {evidence_path}"]
    for item in rows:
        links = item.get("links", {}) or {}
        props = item.get("properties", {}) or {}
        lines.append(f"Entity: {text(props, 'name') or item.get('caption', '?')} ({item.get('schema', '')})")
        lines.append(f"  URL: {links.get('ui', '')}")
        lines.append(f"  Aleph ID: {item.get('id', '')}")
        lines.append("")

    if len(lines) == 1:
        lines.append("No Aleph results found.")
    return "\n".join(lines)


## The Agent

This cell is like `AGENTS.md` plus a simple orchestrator.


In [ ]:
# Prompt: Create a code cell that builds a Pydantic AI agent with YC World, Aleph, and web search tools, and tells it to use registry-discovered names as public-search targets.

# Concise instructions usually work better than long narrative prompts.
# Tool docstrings carry source-specific guidance.

AGENT_INSTRUCTIONS = """
You are a cautious OSINT research assistant.

Goal:
Answer the user's company-research question using source-led evidence.

Workflow:
1. Start with the provided YC World registry preflight; it includes search results and full entity detail from /get-entity.
2. Use YC World relation details and relation counts for founders, directors, owners, authorized signatories, employment rows, and related entities when available. Do not omit authorized signatories just because they are not shareholders.
3. If yc_world_lookup returns a promising YC World ID, use yc_world_entity_detail for deeper relation rows when needed.
4. Extract owners, directors, authorized signatories, partners, and related companies from registry tool output and use those names as public-search targets.
5. Use the custom tools according to their docstrings.
6. Use the provider-native WebSearchTool for journalistic articles, investigations, media reports, NGO reports, public records, and company websites.
7. Before writing public_findings, search the company name plus sanctions or Russia.
8. Do not treat search snippets alone as confirmed evidence; prefer sources with URLs or readable source context.
9. Use at least two source types when possible.

Evidence rules:
- Treat matches as leads unless directly confirmed by a source.
- Separate registry records from public findings.
- Include URLs, source IDs, record IDs, database names, and the consolidated registry_evidence.json path in the source ledger.
- Say clearly when public search found no relevant results.
- Do not include personal national ID numbers in the final report.
- Fill only registry_records, public_findings, and source_ledger.
"""

agent = Agent(
    OpenAIResponsesModel(OPENAI_MODEL),
    output_type=ResearchReport,
    # Provider-native web search, not local DuckDuckGo.
    builtin_tools=[WebSearchTool()],
    tools=[yc_world_lookup, yc_world_entity_detail, aleph_lookup],
    instructions=AGENT_INSTRUCTIONS,
    model_settings=ModelSettings(temperature=0),
)


## Run It

This cell is like the **case file**.

Keep the run prompt short. The agent already has the workflow and evidence rules above, so the prompt only needs the target and the question.


In [ ]:
# Prompt: Create a code cell defining runner functions that execute the agent, save a Markdown report and evidence folder.

def save_report(company: str, question: str, report: ResearchReport) -> Path:
    folder = case_folder(company)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / "report.md"

    sections = [
        ("Registry Records", report.registry_records),
        ("Public Findings", report.public_findings),
        ("Source Ledger", report.source_ledger),
    ]

    lines = [
        f"# OSINT report: {company}",
        "",
        f"Question: {question}",
        "",
    ]

    for title, items in sections:
        if items:
            lines.extend([f"## {title}", "", *[f"- {item}" for item in items], ""])

    path.write_text("\n".join(lines), encoding="utf-8")
    return path


async def run_investigation(company: str, question: str, country: str = "") -> ResearchReport:
    evidence_path = set_case_evidence_file(company)
    country_code = (country or "").strip().lower()
    yc_world_preflight = yc_world_lookup(company, schema="Company", country=country_code)

    prompt = f"""
Target company: {company}
Country: {country_code or "unknown"}
Research question: {question}

Required YC World registry preflight:
{yc_world_preflight}

Use the YC World preflight when answering ownership or control questions. Put concrete
company IDs, incorporation dates, relation roles, relation dates, source URLs, YC World IDs,
and the registry_evidence.json path in registry_records or source_ledger. Include directorship, ownership,
employment, and authorized-signatory relation rows for the exact target company.

Use the names of owners, directors, authorized signatories, partners, and related companies
from the registry preflight as public-search targets. Search for journalistic articles,
investigations, media reports, NGO reports, public records, and company websites.
Before writing public_findings, search the company name plus sanctions or Russia. Put
public-search results in public_findings.

Do not include personal national ID numbers in the final report.
If the YC World preflight is disabled, empty, or inconclusive, say that in registry_records or source_ledger.

Produce a concise source-led report with only registry_records, public_findings, and source_ledger.
"""
    result = await agent.run(prompt)
    report = result.output
    report_path = save_report(company, question, report)
    print(f"Saved report: {report_path}")
    print(f"Evidence JSON: {evidence_path}")

    from IPython.display import Markdown, display
    display(Markdown(report_path.read_text(encoding="utf-8")))
    return report


In [ ]:
# Prompt: Create a code cell with one example investigation call for a company name, country, and research question.

report = await run_investigation(
    company="Consteel Electronics sp. z o.o.",
    country="PL",
    question="Who owns or controls this company, and are there any risk-relevant public claims or leads?"
)
